# 🇾🇪 Yemen Food & Fuel Price Forecasting & Vulnerability Clustering
### Production-Grade Machine Learning & Spatial Market Intelligence Pipeline
**Author & Organization:** Applied Data Science & Humanitarian Intelligence Initiative  
**Data Source:** United Nations World Food Programme (WFP) / Humanitarian Data Exchange (HDX)

---
### 📌 Executive Overview
This notebook implements an end-to-end, production-grade Machine Learning system designed to analyze and forecast market dynamics across 22 Yemeni governorates. The system addresses critical food security challenges by combining:
1. **Unsupervised Market Clustering**: Categorizing regional markets into vulnerability tiers using K-Means and PCA.
2. **Anomaly Detection**: Identifying historical price shock regimes via `IsolationForest`.
3. **Temporal Price Forecasting**: Benchmarking Ridge, Random Forest, and LightGBM models with strict temporal splits to eliminate lookahead leakage.
4. **Production Persistence**: Exporting production-ready pipelines and JSON metadata contracts.

## ⚙️ Cell 1: Environment Setup & Core Dependencies
We install and import modern, reproducible data science and machine learning libraries:
* `pandas` & `numpy` for data manipulation and vectorized calculations.
* `scikit-learn` for preprocessing (`ColumnTransformer`, `StandardScaler`, `OneHotEncoder`), clustering (`KMeans`), anomaly detection (`IsolationForest`), and linear baselines (`Ridge`).
* `lightgbm` for high-performance gradient boosting on tabular market data.
* `seaborn` & `matplotlib` for publication-standard figures.
* `joblib` & `json` for pipeline artifact serialization.

In [ ]:
# Cell 1: Dependencies, configuration, and reproducibility seeds
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Modeling & Evaluation
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score
import lightgbm as lgb

# Plot aesthetics and warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Global Random State for Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Create local directories for artifacts and figures
os.makedirs("artifacts", exist_ok=True)
os.makedirs("figures", exist_ok=True)
os.makedirs("data/raw", exist_ok=True)

print("✅ Setup complete! Environment initialized and directories verified.")
print(f"Python: {sys.version.split()[0]} | NumPy: {np.__version__} | Pandas: {pd.__version__} | LightGBM: {lgb.__version__}")


## 📥 Cell 2: Automated Data Ingestion from UN WFP / HDX
We fetch the official WFP food and fuel price time-series directly from the United Nations Humanitarian Data Exchange (HDX).
The ingestion logic:
1. Connects with robust HTTP streaming and timeouts.
2. Supports HDX direct endpoints with automatic failover to the active dataset mirror.
3. Caches a raw copy in `data/raw/wfp_food_prices_yem.csv` for reproducibility.

In [ ]:
# Cell 2: Data Ingestion from UN HDX WFP Yemen Dataset
import requests

DATA_URLS = [
    "https://data.humdata.org/dataset/98574e10-6866-4f13-a2b1-3c8312801af5/resource/2f0c1d18-d42d-4e43-b928-1bd1fc7f6c90/download/wfp_food_prices_yem.csv",
    "https://data.humdata.org/dataset/07137f6a-4d74-4b5a-93e5-3c0765c71d60/resource/35a09e02-4d56-42d9-a790-ee814fe4f738/download/wfp_food_prices_yem.csv"
]
LOCAL_RAW_PATH = "data/raw/wfp_food_prices_yem.csv"

def download_dataset():
    if os.path.exists(LOCAL_RAW_PATH) and os.path.getsize(LOCAL_RAW_PATH) > 100000:
        print(f"📁 Using cached dataset at {LOCAL_RAW_PATH} ({os.path.getsize(LOCAL_RAW_PATH) / 1024**2:.2f} MB)")
        return
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) WFP-Yemen-Research/1.0"}
    for url in DATA_URLS:
        try:
            print(f"🌐 Fetching live WFP Yemen dataset from: {url}")
            response = requests.get(url, headers=headers, stream=True, timeout=30)
            if response.status_code == 200:
                with open(LOCAL_RAW_PATH, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                print(f"✅ Successfully downloaded {os.path.getsize(LOCAL_RAW_PATH) / 1024**2:.2f} MB to {LOCAL_RAW_PATH}")
                return
            else:
                print(f"⚠️ Status {response.status_code} received from endpoint, trying alternate mirror...")
        except Exception as e:
            print(f"⚠️ Error accessing {url}: {e}")
    raise RuntimeError("Failed to download dataset from all HDX mirrors.")

download_dataset()

# Inspect raw lines
with open(LOCAL_RAW_PATH, 'r', encoding='utf-8') as f:
    sample_lines = [f.readline().strip() for _ in range(3)]
print("\nSample Raw Headers:")
for i, line in enumerate(sample_lines):
    print(f"  Line {i+1}: {line}")


## 🧹 Cell 3: Data Audit, HXL Tag Stripping & Type Sanitization
Raw humanitarian datasets often include Humanitarian Exchange Language (HXL) tag rows (e.g. `#date`, `#adm1+name`) and non-numeric price indicators.
In this step, we:
1. Load the CSV and detect/strip any HXL tagging rows.
2. Parse `date` into a standard datetime object (`pd.to_datetime`).
3. Convert `price` and `usdprice` to clean `float64` numerical values.
4. Filter invalid or non-positive price records ($price > 0$).
5. Audit missing values, date ranges, and unique administrative/commodity coverage.

In [ ]:
# Cell 3: Data Audit and Robust Cleaning
df_raw = pd.read_csv(LOCAL_RAW_PATH, low_memory=False)

print(f"Initial raw record count: {len(df_raw):,} rows, {df_raw.shape[1]} columns")

# 1. Filter out HXL tags if present (rows where date or admin1 starts with '#')
clean_mask = ~df_raw['date'].astype(str).str.startswith('#')
df = df_raw[clean_mask].copy()

# 2. Convert and sanitize date column
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date']).copy()

# 3. Numeric conversions for price fields
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['usdprice'] = pd.to_numeric(df['usdprice'], errors='coerce')
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

# 4. Filter strictly positive prices
initial_count = len(df)
df = df[df['price'] > 0].copy()
dropped_nonpos = initial_count - len(df)

# Sort chronologically by date
df = df.sort_values('date').reset_index(drop=True)

print(f"Cleaned dataset: {len(df):,} valid price records ({dropped_nonpos} non-positive dropped)")
print(f"Date Range: {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}")
print(f"Governorates (admin1): {df['admin1'].nunique()} unique regions")
print(f"Markets: {df['market'].nunique()} markets across {df['admin2'].nunique()} districts")
print(f"Tracked Commodities: {df['commodity'].nunique()} items across {df['category'].nunique()} categories")
print(f"\nPrice Summary in YER:\n{df['price'].describe().to_string()}")


## 🧠 Cell 4: Domain Feature Engineering & Hyperinflation Skew Mitigation
In conflict and high-inflation economic settings such as Yemen, raw currency values suffer from extreme right-skew (ranging from ~24 YER up to 500,000 YER).
To ensure robust statistical modeling and prevent gradient explosion:
1. **Target Transformation**: We model $\log(1 + \text{price})$ (`log1p`), stabilizing residual variance across time.
2. **Temporal Decompositions**: Extract `year`, `month`, `day`, `quarter`, and `dayofweek`.
3. **Cyclical Calendar Features**: Construct sine/cosine cyclical harmonic encodings for month:
   $$\text{month\_sin} = \sin\left(\frac{2\pi \cdot \text{month}}{12}\right), \quad \text{month\_cos} = \cos\left(\frac{2\pi \cdot \text{month}}{12}\right)$$
4. **Historical Trend Variable**: A continuous chronological index (`trend_step = (year - 2009) * 12 + month`) capturing long-term economic shifts.

In [ ]:
# Cell 4: Domain Feature Engineering
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek

# Cyclical seasonal transformation for month (1 to 12)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)

# Long-term continuous inflation/time trend index
df['trend_step'] = (df['year'] - df['year'].min()) * 12 + df['month']

# Log1p transformation of the target price to mitigate inflation skewness
df['log_price'] = np.log1p(df['price'])

print("Feature Engineering Completed:")
print(f"Engineered columns: ['year', 'month', 'quarter', 'day', 'dayofweek', 'month_sin', 'month_cos', 'trend_step', 'log_price']")
print(f"Price Skewness (Raw): {df['price'].skew():.2f} ➔ After log1p: {df['log_price'].skew():.2f} (Near Normal Distribution!)")

# Display distribution summary
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.histplot(df['price'], bins=50, kde=True, color='crimson')
plt.title("Raw Price Distribution (Severe Right Skew)", fontsize=13, fontweight='bold')
plt.xlabel("Price (YER)")
plt.xlim(0, df['price'].quantile(0.99))

plt.subplot(1, 2, 2)
sns.histplot(df['log_price'], bins=50, kde=True, color='teal')
plt.title("log1p(Price) Distribution (Stabilized Variance)", fontsize=13, fontweight='bold')
plt.xlabel("log(1 + Price)")
plt.tight_layout()
plt.savefig("figures/price_transformation.png", dpi=300)
plt.show()


## 🔍 Cell 5: Unsupervised Market Vulnerability Clustering & 2D PCA Profiling
To understand spatial market fragility without assuming predefined labels, we implement an Unsupervised Machine Learning layer:
1. **Market Behavioral Aggregation**: Aggregate market-level time series into operational metrics:
   * `mean_price`: Baseline cost of basket.
   * `std_price`: Price volatility / inflation risk.
   * `cv_price`: Coefficient of variation ($\sigma / \mu$) capturing shock exposure.
   * `max_price`: Peak observed price ceiling.
   * `records_count`: Market reporting liquidity/density.
2. **Standardization**: Scale multi-dimensional features using `StandardScaler`.
3. **Elbow Curve & Silhouette Scoring**: Evaluate cluster separation across $K \in [2, 6]$.
4. **K-Means Clustering ($K=3$)**: Segment regional markets into distinct operational vulnerability profiles:
   * **Cluster 0**: *Stable / Lower Volatility Regional Markets*
   * **Cluster 1**: *Moderate Volatility Urban & Transit Hubs*
   * **Cluster 2**: *High-Vulnerability / Acute Food Shock Zones*
5. **PCA Dimensionality Reduction**: Project high-dimensional market vectors onto 2 principal components and export `figures/pca_clusters.png`.

In [ ]:
# Cell 5: Unsupervised Clustering (K-Means & PCA Market Profiling)
market_features = df.groupby(['admin1', 'market']).agg(
    mean_price=('price', 'mean'),
    std_price=('price', 'std'),
    max_price=('price', 'max'),
    median_price=('price', 'median'),
    records_count=('price', 'count')
).reset_index()

# Impute zero for markets with single record std
market_features['std_price'] = market_features['std_price'].fillna(0)
# Coefficient of Variation
market_features['cv_price'] = market_features['std_price'] / (market_features['mean_price'] + 1e-5)

cluster_cols = ['mean_price', 'std_price', 'cv_price', 'max_price', 'records_count']
X_cluster = market_features[cluster_cols].values

# Scale features
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

# Evaluate K with Elbow (Inertia) & Silhouette Scores
k_range = range(2, 7)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_cluster_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_cluster_scaled, labels))

print("K-Means Cluster Diagnostics (K=2..6):")
for k, s, in_val in zip(k_range, silhouettes, inertias):
    print(f"  K={k} ➔ Silhouette Score: {s:.4f} | Inertia: {in_val:.2f}")

# Train optimal K-Means model (K=3)
optimal_k = 3
kmeans_model = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE, n_init=10)
market_features['cluster'] = kmeans_model.fit_predict(X_cluster_scaled)

# PCA 2D Projection
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_comps = pca.fit_transform(X_cluster_scaled)
market_features['pca_1'] = pca_comps[:, 0]
market_features['pca_2'] = pca_comps[:, 1]

print(f"\nPCA Explained Variance Ratio: PC1={pca.explained_variance_ratio_[0]:.2%}, PC2={pca.explained_variance_ratio_[1]:.2%} (Total={sum(pca.explained_variance_ratio_):.2%})")

# Visualizing PCA Clusters
plt.figure(figsize=(11, 7))
cluster_palette = {0: '#2b5c8f', 1: '#d95f02', 2: '#7570b3'}

scatter = sns.scatterplot(
    data=market_features,
    x='pca_1', y='pca_2',
    hue='cluster',
    palette=cluster_palette,
    s=140, edgecolor='black', alpha=0.9
)

# Label top representative markets on PCA plot
for _, row in market_features.iterrows():
    if row['pca_1'] > 2.0 or row['pca_2'] > 2.0 or row['cluster'] in [1, 2]:
        plt.text(row['pca_1'] + 0.1, row['pca_2'], f"{row['market']} ({row['admin1']})", fontsize=8.5, alpha=0.85)

plt.title("Yemen Market Vulnerability Profiles (2D PCA of Regional Dynamics)", fontsize=13, fontweight='bold')
plt.xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]:.1%} Variance)")
plt.ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]:.1%} Variance)")
plt.legend(title="Vulnerability Tier", loc="upper right")
plt.tight_layout()
plt.savefig("figures/pca_clusters.png", dpi=300)
plt.show()

# Display Cluster Financial Profiles in YER
cluster_profiles = market_features.groupby('cluster').agg(
    market_count=('market', 'count'),
    avg_price_yer=('mean_price', 'mean'),
    avg_volatility_yer=('std_price', 'mean'),
    avg_cv=('cv_price', 'mean'),
    avg_records=('records_count', 'mean')
).reset_index()
print("\n--- Cluster Financial & Operational Profiles (YER) ---")
print(cluster_profiles.to_string(index=False))

# Persist unsupervised clustering pipeline
clustering_artifacts = {
    'scaler': scaler_cluster,
    'kmeans': kmeans_model,
    'pca': pca,
    'features': cluster_cols,
    'market_profiles': market_features[['admin1', 'market', 'cluster', 'pca_1', 'pca_2']]
}
joblib.dump(clustering_artifacts, "artifacts/market_clustering_pipeline.joblib")
print("✅ Clustering pipeline persisted to artifacts/market_clustering_pipeline.joblib")

# Map clusters back to primary dataframe as operational feature
market_cluster_map = market_features.set_index(['admin1', 'market'])['cluster'].to_dict()
df['market_cluster'] = df.apply(lambda r: market_cluster_map.get((r['admin1'], r['market']), 0), axis=1)


## 🚨 Cell 6: Unsupervised Anomaly Detection via IsolationForest
In crisis economies, hyperinflation surges, currency devaluations, and blockades trigger acute price shock events.
In this step, we:
1. Apply `IsolationForest` (contamination=0.03, estimating ~3% historical systemic shocks).
2. Compute market anomaly scores on standardized $(\text{price}, \text{usdprice}, \text{trend\_step})$ vectors.
3. Classify records into `is_anomaly` ($-1 = \text{Price Shock}, 1 = \text{Normal Trend}$).
4. Identify the historical shock clusters (such as major currency collapses in Aden/Sana'a in 2021-2022).
5. Visualize the detected shocks against the chronological timeline.

In [ ]:
# Cell 6: IsolationForest Anomaly Detection for Historical Market Shocks
anomaly_features = ['price', 'usdprice', 'trend_step']
df_anomaly = df.dropna(subset=anomaly_features).copy()

iso_scaler = StandardScaler()
X_iso = iso_scaler.fit_transform(df_anomaly[anomaly_features])

# Contamination parameter set to 0.03 (identifying top 3% extreme economic shocks)
iso_forest = IsolationForest(contamination=0.03, random_state=RANDOM_STATE, n_estimators=100)
df_anomaly['anomaly_label'] = iso_forest.fit_predict(X_iso)
df_anomaly['anomaly_score'] = iso_forest.decision_function(X_iso)

# Map back to main df
df['anomaly_label'] = 1
df.loc[df_anomaly.index, 'anomaly_label'] = df_anomaly['anomaly_label']
df['anomaly_score'] = 0.0
df.loc[df_anomaly.index, 'anomaly_score'] = df_anomaly['anomaly_score']

num_anomalies = (df['anomaly_label'] == -1).sum()
pct_anomalies = (num_anomalies / len(df)) * 100
print(f"IsolationForest Audit: {num_anomalies:,} price shock anomalies detected ({pct_anomalies:.2f}% of dataset)")

# Historical peak anomaly dates
top_shock_dates = df[df['anomaly_label'] == -1].groupby('year').size().reset_index(name='anomaly_count')
print("\nHistorical Distribution of Price Shock Anomalies:")
print(top_shock_dates.to_string(index=False))

# Plot historical time-series with highlighted price shock anomalies
plt.figure(figsize=(14, 6))
sample_market = df[df['commodity'].isin(['Wheat flour', 'Diesel', 'Petrol'])].sort_values('date')
sns.lineplot(data=sample_market, x='date', y='price', hue='commodity', alpha=0.5, legend='brief')
anom_pts = sample_market[sample_market['anomaly_label'] == -1]
plt.scatter(anom_pts['date'], anom_pts['price'], color='red', s=35, label='Detected Price Shock Anomaly', zorder=5)

plt.title("Historical Food & Fuel Price Trajectory with IsolationForest Anomalies", fontsize=13, fontweight='bold')
plt.xlabel("Date")
plt.ylabel("Price (YER)")
plt.yscale('log')
plt.legend(loc="upper left")
plt.tight_layout()
plt.savefig("figures/historical_anomalies.png", dpi=300)
plt.show()


## ⏳ Cell 7: Split-First Temporal Cutoff Protocol (Zero Data Leakage)
In time-series market forecasting, standard random cross-validation creates severe **data leakage** (future prices leaking into past predictions, causing artificially inflated $R^2$ scores).
To ensure production-grade validity:
1. **Strict Temporal Boundary**:
   * **Training Partition**: All historical observations prior to $2023$ ($date < 2023\text{-}01\text{-}01$).
   * **Test / Out-of-Sample Partition**: All observations from $2023$ onwards ($date \ge 2023\text{-}01\text{-}01$).
2. **Feature Integrity**: The test set is completely held out during all feature scalers, encoders, and model training.
3. **Target Variable**: We predict $\log(1 + \text{price})$ (`log_price`) and evaluate on both log scale and true Yemeni Rial (`expm1(log_price) = price`).

In [ ]:
# Cell 7: Temporal Split Protocol (Leakage Prevention)
TEMPORAL_CUTOFF_DATE = "2023-01-01"

train_mask = df['date'] < TEMPORAL_CUTOFF_DATE
test_mask = df['date'] >= TEMPORAL_CUTOFF_DATE

df_train = df[train_mask].copy()
df_test = df[test_mask].copy()

# Feature column definitions
numeric_features = [
    'year', 'month', 'quarter', 'day', 'dayofweek',
    'month_sin', 'month_cos', 'trend_step',
    'latitude', 'longitude', 'market_cluster'
]

categorical_features = [
    'admin1', 'admin2', 'market',
    'category', 'commodity', 'unit', 'pricetype'
]

target_col = 'log_price'
raw_target_col = 'price'

X_train = df_train[numeric_features + categorical_features]
y_train = df_train[target_col]
y_train_raw = df_train[raw_target_col]

X_test = df_test[numeric_features + categorical_features]
y_test = df_test[target_col]
y_test_raw = df_test[raw_target_col]

print(f"--- Temporal Split Verification (Cutoff: {TEMPORAL_CUTOFF_DATE}) ---")
print(f"Train Records: {len(X_train):,} ({len(X_train)/len(df):.1%}) | Range: {df_train['date'].min().strftime('%Y-%m-%d')} to {df_train['date'].max().strftime('%Y-%m-%d')}")
print(f"Test Records:  {len(X_test):,} ({len(X_test)/len(df):.1%}) | Range: {df_test['date'].min().strftime('%Y-%m-%d')} to {df_test['date'].max().strftime('%Y-%m-%d')}")
print(f"Total Features: {len(numeric_features) + len(categorical_features)} ({len(numeric_features)} numeric, {len(categorical_features)} categorical)")
assert df_train['date'].max() < df_test['date'].min(), "CRITICAL: Temporal overlap detected!"
print("✅ Leakage Check Passed: Max Train Date strictly precedes Min Test Date.")


## 🏗️ Cell 8: Unified Preprocessing Pipeline (ColumnTransformer)
To ensure production-grade deployment with zero lookahead bias:
1. **Numeric Pipeline**:
   * `SimpleImputer(strategy='median')`: Imputes missing coordinates or time steps.
   * `StandardScaler()`: Standardizes scale across continuous numeric features.
2. **Categorical Pipeline**:
   * `SimpleImputer(strategy='most_frequent')`: Robust imputation for categorical attributes.
   * `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`: Encodes commodities, administrative regions, markets, and units without failing on unseen test categories.
3. **ColumnTransformer Integration**:
   * Merges numeric and categorical pipelines into a single composable preprocessing estimator.

In [ ]:
# Cell 8: Supervised Preprocessing & Pipeline Construction
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

# Fit on training partition only
preprocessor.fit(X_train)

# Transform a batch to verify transformed dimensions
X_train_trans = preprocessor.transform(X_train)
X_test_trans = preprocessor.transform(X_test)

print("Unified ColumnTransformer Pipeline Built:")
print(f"Original feature count: {len(numeric_features) + len(categorical_features)}")
print(f"Transformed feature matrix: {X_train_trans.shape[1]} one-hot encoded dimensions")
print(f"Train matrix shape: {X_train_trans.shape} | Test matrix shape: {X_test_trans.shape}")


## 📊 Cell 9: Model Benchmarking & Multi-Algorithm Experiment Matrix
We benchmark 3 diverse model families trained on identical temporal splits:
1. **L2-Regularized Linear Baseline**: `Ridge(alpha=1.0)`
2. **Bagging Ensemble**: `RandomForestRegressor(n_estimators=100, max_depth=16, random_state=42)`
3. **Gradient Boosting Machine**: `LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42)`

**Evaluation Protocols**:
* **Log-scale Metrics**: $R^2$ on the target $\log(1 + \text{price})$.
* **Real Financial Scale Metrics (YER)**: We invert predictions via $\hat{y}_{\text{raw}} = \exp(\hat{y}_{\text{log}}) - 1$ and measure:
  * **Mean Absolute Error (MAE)** in Yemeni Rial.
  * **Root Mean Squared Error (RMSE)** in Yemeni Rial.
* Tabulate results in an audit-ready comparison table and plot real vs. predicted price benchmarks.

In [ ]:
# Cell 9: Model Benchmarking & Experiment Matrix
import time

candidate_models = {
    'Ridge (L2 Linear)': Ridge(alpha=10.0, random_state=RANDOM_STATE),
    'Random Forest (Bagging)': RandomForestRegressor(n_estimators=100, max_depth=16, random_state=RANDOM_STATE, n_jobs=-1),
    'LightGBM (Boosting)': lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
}

benchmark_results = []
trained_pipelines = {}

for name, model in candidate_models.items():
    print(f"⏳ Fitting pipeline for {name}...")
    start_time = time.time()

    # Construct complete pipeline
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # Fit pipeline
    pipeline.fit(X_train, y_train)
    fit_duration = time.time() - start_time

    # Predict on test set
    y_pred_log = pipeline.predict(X_test)

    # Real financial scale (YER) via expm1
    y_pred_raw = np.expm1(y_pred_log)

    # Calculate metrics
    mae_yer = mean_absolute_error(y_test_raw, y_pred_raw)
    rmse_yer = np.sqrt(mean_squared_error(y_test_raw, y_pred_raw))
    r2_log = r2_score(y_test, y_pred_log)

    trained_pipelines[name] = {
        'pipeline': pipeline,
        'y_pred_log': y_pred_log,
        'y_pred_raw': y_pred_raw,
        'mae_yer': mae_yer,
        'rmse_yer': rmse_yer,
        'r2_log': r2_log,
        'fit_time': fit_duration
    }

    benchmark_results.append({
        'Model': name,
        'MAE (YER)': round(mae_yer, 2),
        'RMSE (YER)': round(rmse_yer, 2),
        'Log R² Score': round(r2_log, 4),
        'Fit Time (s)': round(fit_duration, 2)
    })
    print(f"  ✓ {name} completed in {fit_duration:.2f}s | MAE: {mae_yer:,.2f} YER | Log R²: {r2_log:.4f}")

# Convert to DataFrame
df_benchmark = pd.DataFrame(benchmark_results)
print("\n" + "="*70)
print("🏆 MODEL BENCHMARKING EXPERIMENT MATRIX (OUT-OF-SAMPLE TEST 2023-2026)")
print("="*70)
print(df_benchmark.to_string(index=False))

# Identify best model based on MAE (YER)
best_model_name = df_benchmark.sort_values(by='MAE (YER)').iloc[0]['Model']
best_pipeline = trained_pipelines[best_model_name]['pipeline']
print(f"\n🌟 Top Performing Model Selected: {best_model_name}")

# Visualization of Predictions
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
# Benchmark metrics comparison bar plot
x_pos = np.arange(len(df_benchmark))
plt.bar(x_pos, df_benchmark['MAE (YER)'], color=['#4c72b0', '#55a868', '#c44e52'], width=0.55)
plt.xticks(x_pos, df_benchmark['Model'], rotation=15, fontsize=10)
plt.title("Out-of-Sample MAE in Yemeni Rial (Lower is Better)", fontsize=12, fontweight='bold')
plt.ylabel("Mean Absolute Error (YER)")

plt.subplot(1, 2, 2)
# True vs Predicted Prices Scatter Plot (Sampled for visual clarity)
sample_idx = np.random.choice(len(y_test_raw), size=min(1500, len(y_test_raw)), replace=False)
y_sample_true = y_test_raw.iloc[sample_idx]
y_sample_pred = trained_pipelines[best_model_name]['y_pred_raw'][sample_idx]

plt.scatter(y_sample_true, y_sample_pred, alpha=0.35, color='darkblue', edgecolors='none', s=25)
max_val = max(y_sample_true.quantile(0.99), y_sample_pred.max())
plt.plot([0, max_val], [0, max_val], 'r--', lw=2, label='Perfect Prediction (1:1)')
plt.xlim(0, max_val)
plt.ylim(0, max_val)
plt.title(f"{best_model_name}: True vs. Predicted (YER)", fontsize=12, fontweight='bold')
plt.xlabel("Actual Price (YER)")
plt.ylabel("Forecasted Price (YER)")
plt.legend()

plt.tight_layout()
plt.savefig("figures/model_benchmarks.png", dpi=300)
plt.show()


## 💾 Cell 10: Model Persistence & JSON Metadata Contract
In enterprise MLOps architectures, serialization must include both the trained estimator pipeline and an audit-ready metadata contract defining feature schemas, dependencies, and performance metrics.
In this step, we:
1. Export the top-performing end-to-end forecasting pipeline to `artifacts/best_forecasting_pipeline.joblib`.
2. Generate an audit-ready `artifacts/metadata.json` capturing:
   * Training date cutoff (`2023-01-01`).
   * Benchmark metrics (MAE, RMSE, Log R²).
   * Feature schema (numeric and categorical feature registries).
   * Runtime environment (Python version, Scikit-learn, LightGBM versions).
3. Test pipeline round-trip serialization by loading `artifacts/best_forecasting_pipeline.joblib` and executing a live inference test on sample input data.

In [ ]:
# Cell 10: Model Persistence & JSON Metadata Contract
MODEL_EXPORT_PATH = "artifacts/best_forecasting_pipeline.joblib"
METADATA_EXPORT_PATH = "artifacts/metadata.json"

# 1. Export best forecasting pipeline
joblib.dump(best_pipeline, MODEL_EXPORT_PATH)
print(f"✅ Successfully exported best forecasting pipeline to {MODEL_EXPORT_PATH} ({os.path.getsize(MODEL_EXPORT_PATH) / 1024**2:.2f} MB)")

# 2. Build metadata contract
metadata_contract = {
    "project_name": "yemen-market-price-intelligence",
    "model_family": best_model_name,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%SZ", time.gmtime()),
    "data_source": "UN WFP / Humanitarian Data Exchange (HDX)",
    "temporal_split_protocol": {
        "cutoff_date": TEMPORAL_CUTOFF_DATE,
        "train_samples": int(len(X_train)),
        "test_samples": int(len(X_test)),
        "train_date_range": [df_train['date'].min().strftime('%Y-%m-%d'), df_train['date'].max().strftime('%Y-%m-%d')],
        "test_date_range": [df_test['date'].min().strftime('%Y-%m-%d'), df_test['date'].max().strftime('%Y-%m-%d')]
    },
    "feature_schema": {
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "total_raw_features": len(numeric_features) + len(categorical_features),
        "target_variable": target_col,
        "target_transformation": "log1p(price_YER)",
        "evaluation_inversion": "expm1(predicted_log_price)"
    },
    "test_metrics_summary": {
        "mae_yer": round(float(trained_pipelines[best_model_name]['mae_yer']), 2),
        "rmse_yer": round(float(trained_pipelines[best_model_name]['rmse_yer']), 2),
        "log_r2_score": round(float(trained_pipelines[best_model_name]['r2_log']), 4)
    },
    "benchmark_comparison": df_benchmark.to_dict(orient='records'),
    "runtime_environment": {
        "python_version": sys.version.split()[0],
        "scikit_learn_version": joblib.__name__,
        "lightgbm_version": lgb.__version__,
        "random_state": RANDOM_STATE
    }
}

# Write metadata.json
with open(METADATA_EXPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata_contract, f, indent=4)
print(f"✅ Audit-ready metadata saved to {METADATA_EXPORT_PATH}")

# 3. Live Round-Trip Verification
print("\n--- Live Inference Verification ---")
reloaded_pipeline = joblib.load(MODEL_EXPORT_PATH)
sample_test_row = X_test.iloc[0:3]
sample_true_yer = y_test_raw.iloc[0:3].values

sample_pred_log = reloaded_pipeline.predict(sample_test_row)
sample_pred_yer = np.expm1(sample_pred_log)

for i in range(len(sample_test_row)):
    item = sample_test_row.iloc[i]['commodity']
    market = sample_test_row.iloc[i]['market']
    true_p = sample_true_yer[i]
    pred_p = sample_pred_yer[i]
    abs_err = abs(true_p - pred_p)
    print(f"Sample {i+1} [{item} in {market}]: Actual: {true_p:,.1f} YER | Forecast: {pred_p:,.1f} YER (Err: {abs_err:,.1f} YER)")

print("\n🎉 End-to-End Workflow Execution Verified Successfully in Google Colab!")
